# Spine XR Augmentation — Project 3 Colab Runner

Google Colab Pro+ (A100) üzerinde koşulmak üzere tasarlandı. Tüm ağır eğitim buradan yürütülür.

## Sıralama

1. Bootstrap (Drive mount, repo + dataset.rar'ı SSD'ye çek, outputs'u Drive'a sembolik bağla, pip install)
2. `01_audit` → `02_data_splitter`
3. `03_train_classifier` — 4 Case × 2 Backbone (VGG16, InceptionV3) — baseline (no aug)
4. (Sonraki milestone'larda) Phase 04 traditional, Phase 05–07 WGAN, Phase 08 hybrid, Phase 09 final report

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adı "dataset" olmalı — configs/base.yaml relatif `dataset/...` yolları kullanır.
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/dataset.rar')
!unrar x -o+ {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr-augmentation-study/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs') and not os.path.islink('outputs'):
    shutil.rmtree('outputs')
elif os.path.islink('outputs'):
    os.remove('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Audit + Splits

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_data_splitter.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/01_audit/audit_report.md
!echo '---'
!cat outputs/02_splits/splits_summary.md

## 3. Baseline classifier — 4 Case × 2 Backbone

### Smoke Test

In [ ]:
# SMOKE: en küçük case (case_4) + VGG16 + 1 epoch — pipeline çalışıyor mu kontrolü
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16 \
    --epochs 1 \
    --out-tag 03_smoke

### Case 1 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_1 \
    --backbones-filter vgg16

### Case 1 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_1 \
    --backbones-filter inception_v3

### Case 2 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_2 \
    --backbones-filter vgg16

### Case 2 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_2 \
    --backbones-filter inception_v3

### Case 3 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_3 \
    --backbones-filter vgg16

### Case 3 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_3 \
    --backbones-filter inception_v3

### Case 4 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16

### Case 4 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter inception_v3

### Sonuçları Özetle (8 hücre tamamlandıktan sonra)

In [ ]:
# Tüm cell'leri tarayıp tek bir summary.md/csv üreten yardımcı betik (sonraki milestone'a kadar elle bu hücre yeter)
import json, pandas as pd
from pathlib import Path

rows = []
for case_dir in sorted(Path('outputs/03_baseline').glob('case_*')):
    for bb_dir in sorted(case_dir.glob('*')):
        m_path = bb_dir / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        row = {
            'case': m['case'], 'backbone': m['backbone'],
            'best_epoch': m['best_epoch'],
            'best_test_macro_f1': round(m['best_test_macro_f1'], 4),
            'best_val_macro_f1': round(m['best_val_macro_f1'], 4) if m['best_val_macro_f1'] == m['best_val_macro_f1'] else float('nan'),
        }
        for c, pc in m['best_test_metrics']['per_class'].items():
            row[f'F1__{c}'] = round(pc['f1'], 4)
        rows.append(row)
df = pd.DataFrame(rows)
Path('outputs/03_baseline').mkdir(parents=True, exist_ok=True)
df.to_csv('outputs/03_baseline/summary.csv', index=False)
Path('outputs/03_baseline/summary.md').write_text('# Baseline summary\n\n' + df.to_markdown(index=False))
df

## 4. Traditional augmentation - 4 Case x 2 Backbone

Plan D6 + paper sec 5.3 best transforms (offline): Rotation 270 + Shearing 30 + per-case 2nd rotation (Rot 90 for Case 1, Rot 45 for Cases 2/3/4). NF satirlarina aug uygulanmaz. internal_val ve test 100 percent gercek, outputs/02_splits/ altindan okunmaya devam eder.

### 4.1 Build the augmented training set

In [ ]:
# Her case icin: abnormal satirlara 3 transform uygula, PNG'leri outputs/04_traditional/<case>/aug_pngs/ altina kaydet,
# train_traditional.csv = real_rows + aug_rows uret. Idempotent - yeniden calistirmak guvenli.
!python scripts/04_build_traditional_set.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/04_traditional/summary.md

### 4.2 Smoke test

In [ ]:
# Phase 04 pipeline'i calisiyor mu? case_4 + VGG16 + 1 epoch.
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --cases-filter case_4 \
    --backbones-filter vgg16 \
    --epochs 1 \
    --out-tag 04_smoke

### Case 1 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_1 \
    --backbones-filter vgg16

### Case 1 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_1 \
    --backbones-filter inception_v3

### Case 2 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_2 \
    --backbones-filter vgg16

### Case 2 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_2 \
    --backbones-filter inception_v3

### Case 3 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_3 \
    --backbones-filter vgg16

### Case 3 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_3 \
    --backbones-filter inception_v3

### Case 4 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_4 \
    --backbones-filter vgg16

### Case 4 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_4 \
    --backbones-filter inception_v3

### Sonuclari Ozetle (8 hucre tamamlandiktan sonra)

In [ ]:
import json, pandas as pd
from pathlib import Path

def collect(out_tag):
    rows = []
    root = Path(f'outputs/{out_tag}')
    for case_dir in sorted(root.glob('case_*')):
        for bb_dir in sorted(case_dir.glob('*')):
            m_path = bb_dir / 'metrics.json'
            if not m_path.exists():
                continue
            m = json.loads(m_path.read_text())
            row = {
                'phase': out_tag, 'case': m['case'], 'backbone': m['backbone'],
                'best_epoch': m['best_epoch'],
                'best_test_macro_f1': round(m['best_test_macro_f1'], 4),
            }
            for c, pc in m['best_test_metrics']['per_class'].items():
                row[f'F1__{c}'] = round(pc['f1'], 4)
            rows.append(row)
    return pd.DataFrame(rows)

df_trad = collect('04_traditional')
df_trad.to_csv('outputs/04_traditional/summary.csv', index=False)
Path('outputs/04_traditional/summary.md').write_text('# Traditional summary\n\n' + df_trad.to_markdown(index=False))

df_base = collect('03_baseline')
if len(df_base) and len(df_trad):
    cmp = df_base[['case','backbone','best_test_macro_f1']].rename(columns={'best_test_macro_f1':'baseline'}).merge(
        df_trad[['case','backbone','best_test_macro_f1']].rename(columns={'best_test_macro_f1':'traditional'}),
        on=['case','backbone'])
    cmp['delta'] = (cmp['traditional'] - cmp['baseline']).round(4)
    print(cmp.to_string(index=False))
df_trad

## 5. WGAN per minority class - paper sec 4.6.2 / plan D5

Per minority class WGAN (DSN, VC, FS, Spondy, Surgical Implant, Other Lesions). Osteophytes excluded - already abnormal-majority. Paper-faithful default = Wasserstein loss with WEIGHT CLIPPING (paper Fig 6 / Sec 4.6.2). Latent 120, image 256x256, n_critic=5, RMSProp lr=5e-5, weight clip 0.01.

WGAN-GP escape hatch available via --loss gp if weight-clip becomes unstable (paper itself reports collapse after epoch 300 - known weight-clip pathology).

Snapshots every 1500 generator iterations after iter 5000. Phase 06 will compute FID per snapshot and pick the deployable checkpoint.

### 5.1 Smoke test (Vertebral collapse, 600 iters)

In [ ]:
# Pipeline calisiyor mu? En kucuk havuzlu sinifta (VC ~139 satir) hizli smoke.
# Iter 600 - 1 snapshot olusturmaz (first_snapshot_at=5000). Sadece pipeline'i ve sample png'leri uretir.
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --classes-filter "Vertebral collapse" \
    --iterations 600 \
    --out-tag 05_smoke
!ls outputs/05_smoke/case_1/vertebral_collapse/samples/ | head -5

### 5.2 Per-class training (paper-faithful, weight_clip)

**Sure tahmini (A100):** Iter basina ~0.4-0.6 sn (n_critic=5 dahil). 20K iter / sinif = ~2-3 saat / sinif. 6 sinif = ~12-18 saat - Colab Pro+ ile birden fazla oturuma yayilabilir.

Her sinif kendi hucresinde - tek bir oturumda da, parcali da kosturulabilir.

#### Disc space narrowing (case_1)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Disc space narrowing"

#### Vertebral collapse (case_1)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Vertebral collapse"

#### Foraminal stenosis (case_2)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Foraminal stenosis"

#### Spondylolysthesis (case_2)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Spondylolysthesis"

#### Surgical implant (case_3)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Surgical implant"

#### Other lesions (case_4)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Other lesions"

### 5.3 Quick QA - sample grids

In [ ]:
# En son uretilen sample grid'leri yan yana goster.
from pathlib import Path
from IPython.display import Image, display, HTML
for case_dir in sorted(Path('outputs/05_wgan').glob('case_*')):
    for cls_dir in sorted(case_dir.glob('*')):
        samples = sorted((cls_dir / 'samples').glob('iter_*.png'))
        if not samples:
            continue
        print(f'--- {case_dir.name}/{cls_dir.name} (last sample: {samples[-1].name}) ---')
        display(Image(str(samples[-1])))


### 5.4 (Optional) WGAN-GP escape hatch

In [ ]:
# Weight-clip patolojik davranirsa (loss patlamasi, sample bozulmasi) ayni kodu GP ile cagir.
# !python scripts/05_train_wgan.py \
#     --config configs/base.yaml \
#     --wgan configs/wgan.yaml \
#     --classes-filter "Vertebral collapse" \
#     --loss gp \
#     --out-tag 05_wgan_gp